# RAG Pipelines - Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
# from langchain_core.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

/var/folders/8y/3b6b9p6d5vjg7ql8yvqrf69w0000gn/T/ipykernel_6809/2700154759.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
/Users/rahulshelke/Documents/Data-Science/Data-Science-Projects/Complete-Agentic-AI-Course/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Read all the pdf's inside directory

In [2]:
def process_all_pdf(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    # find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {str(pdf_file)}")
        try:
            loader = PyMuPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'

            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

In [3]:
# process all pdf documents in the data directory
all_pdf_documents = process_all_pdf("../data/pdf/")

Found 2 PDF files to process

Processing: ../data/pdf/designing-machine-learning-systems.pdf
✅ Loaded 501 pages

Processing: ../data/pdf/attention.pdf
✅ Loaded 15 pages

Total documents loaded: 516


### 2. Text Splitting get into Chunks

In [4]:
def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [5]:
chunks = split_documents(all_pdf_documents)

Split 516 documents into 1234 chunks

Example chunk:
Content: Praise for Designing Machine Learning Systems
There is so much information one needs to know to be an effective
machine learning engineer. It’s hard to cut through the chaff to get the
most relevant i...
Metadata: {'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2024-06-03T00:57:21+00:00', 'source': '../data/pdf/designing-machine-learning-systems.pdf', 'file_path': '../data/pdf/designing-machine-learning-systems.pdf', 'total_pages': 501, 'format': 'PDF 1.5', 'title': 'Designing Machine Learning Systems', 'author': 'Chip Huyen', 'subject': '', 'keywords': '', 'moddate': '2024-06-03T00:57:31+00:00', 'trapped': '', 'modDate': "D:20240603005731+00'00'", 'creationDate': "D:20240603005721+00'00'", 'page': 1, 'source_file': 'designing-machine-learning-systems.pdf', 'file_type': 'pdf'}


In [6]:
chunks

[Document(metadata={'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2024-06-03T00:57:21+00:00', 'source': '../data/pdf/designing-machine-learning-systems.pdf', 'file_path': '../data/pdf/designing-machine-learning-systems.pdf', 'total_pages': 501, 'format': 'PDF 1.5', 'title': 'Designing Machine Learning Systems', 'author': 'Chip Huyen', 'subject': '', 'keywords': '', 'moddate': '2024-06-03T00:57:31+00:00', 'trapped': '', 'modDate': "D:20240603005731+00'00'", 'creationDate': "D:20240603005721+00'00'", 'page': 1, 'source_file': 'designing-machine-learning-systems.pdf', 'file_type': 'pdf'}, page_content='Praise for Designing Machine Learning Systems\nThere is so much information one needs to know to be an effective\nmachine learning engineer. It’s hard to cut through the chaff to get the\nmost relevant information, but Chip has done that admirably with this\nbook. If you are serious about ML in production, and care about how to\ndesign and implement ML systems end to end, this b

## 3. Embedding

In [7]:
import numpy as np
from sentence_transformers import SentenceTransformer # embedding model from here
import chromadb
from chromadb.config import Settings
import uuid # unique record id
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity # while retrival


In [8]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager

        Args:
            model_name: Huggingface model name for sentence embeddings
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {0}")
            raise e

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

    def get_embedding_dimension(self) -> int:
        """Get the embedding dimension of the model"""
        if not self.model:
            raise ValueError("Model not loaded")
        return self.model.get_embedding_dimension()

### Initalize Embedding Manager

In [9]:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6434.71it/s]


Model loaded successfully. Embedding dimension: 384


## 4. VectorStoreDB

In [10]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name:str = "pdf_documents",
        persist_directory:str = "../data/vector_store"
        ):
        """
        Initalize the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initalize_store()

    def _initalize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"descriptio": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initalized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initalizing vector error: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vactor store 

        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for documents
        """
        try:
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match the number of embeddings")

            print(f"Adding {len(documents)} documents to vector store...")

            # Prepare data for chromadb
            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []

            for i , (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)

                # prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)

                # Document content
                documents_text.append(doc.page_content)

                # Embedding
                embeddings_list.append(embedding.tolist())

            # add collection
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error adding documents error: {e}")

In [11]:
vectorstore = VectorStore()
vectorstore

Vector store initalized. Collection: pdf_documents
Existing documents in collection: 2468


In [12]:
chunks

[Document(metadata={'producer': 'ConvertAPI', 'creator': '', 'creationdate': '2024-06-03T00:57:21+00:00', 'source': '../data/pdf/designing-machine-learning-systems.pdf', 'file_path': '../data/pdf/designing-machine-learning-systems.pdf', 'total_pages': 501, 'format': 'PDF 1.5', 'title': 'Designing Machine Learning Systems', 'author': 'Chip Huyen', 'subject': '', 'keywords': '', 'moddate': '2024-06-03T00:57:31+00:00', 'trapped': '', 'modDate': "D:20240603005731+00'00'", 'creationDate': "D:20240603005721+00'00'", 'page': 1, 'source_file': 'designing-machine-learning-systems.pdf', 'file_type': 'pdf'}, page_content='Praise for Designing Machine Learning Systems\nThere is so much information one needs to know to be an effective\nmachine learning engineer. It’s hard to cut through the chaff to get the\nmost relevant information, but Chip has done that admirably with this\nbook. If you are serious about ML in production, and care about how to\ndesign and implement ML systems end to end, this b

### Convert the text to embeddings

In [13]:
texts = [doc.page_content for doc in chunks]

# texts

# generate the embeddings

embeddings = embedding_manager.generate_embeddings(texts)

## store into vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 1234 texts...


Batches: 100%|██████████| 39/39 [00:05<00:00,  6.87it/s]


Generated embeddings with shape: (1234, 384)
Adding 1234 documents to vector store...
Successfully added 1234 documents to vector store
Total documents in collection: 3702


# Retriver Pipeline From VectorStore

In [14]:
class RAGRetriever:
    """Handles query-based retrieval from the vectro store"""
    
    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initalize the retriver

        Args:
            vector_store: vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retriever(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query

        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold

        Returns:
            List of directionaries containing retrived documents and metadata
        """
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, Score threshold: {score_threshold}")

        # generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # COnvert distance to similarity score (ChromaDB uses consine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'ids': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                print(f"Retrived {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No document found")

            return retrieved_docs

        except Exception as e:
            print(f"Errod during retrieval: {e}")
            raise[]

In [15]:
rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [16]:
rag_retriever

In [17]:
rag_retriever.retriever("what is dataflow ?")

Retrieving documents for query: 'what is dataflow ?'
Top k: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 19.86it/s]

Generated embeddings with shape: (1, 384)
Retrived 5 documents (after filtering)


[{'ids': 'doc_9f189e6d_240',
  'content': 'modes of dataflow:\nData passing through databases\nData passing through services using requests such as the requests\nprovided by REST and RPC APIs (e.g., POST/GET requests)\nData passing through a real-time transport like Apache Kafka and\nAmazon Kinesis\nWe’ll go over each of them in this section.\nData Passing Through Databases\nThe easiest way to pass data between two processes is through databases,\nwhich we’ve discussed in the section “Data Storage Engines and\nProcessing”. For example, to pass data from process A to process B, process\nA can write that data into a database, and process B simply reads from that\ndatabase.',
  'metadata': {'trapped': '',
   'producer': 'ConvertAPI',
   'creator': '',
   'doc_index': 240,
   'format': 'PDF 1.5',
   'page': 103,
   'subject': '',
   'title': 'Designing Machine Learning Systems',
   'file_type': 'pdf',
   'content_length': 624,
   'keywords': '',
   'creationdate': '2024-06-03T00:57:21+00:0

# Integration Vectordb Context Pipeline with LLM output

### Simple RAG Pipeline with Groq LLM

In [18]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

True

### 1. Initialize the Groq LLM (set your GROQ_API_KEY in environment)

In [19]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [20]:
llm = ChatGroq(
    model_name = "qwen/qwen3.6-27b",
    temperature=0.1,
    max_tokens=1024
)

In [21]:
llm.invoke("hi")

AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user said "hi". This is a simple greeting.\n2.  **Identify Intent:** The user is initiating a conversation. No specific question or task is provided.\n3.  **Determine Response Strategy:** \n   - Acknowledge the greeting warmly.\n   - Keep it open-ended to encourage the user to share what they need.\n   - Maintain a friendly, helpful tone.\n4.  **Draft Response:** "Hi there! How can I help you today?" \n5.  **Refine Response:** The draft is concise, polite, and invites further interaction. It matches standard AI assistant behavior.\n6.  **Final Output Generation:** Output the refined response.✅\n</think>\n\nHi there! How can I help you today? 😊', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 11, 'total_tokens': 187, 'completion_time': 0.33349991, 'completion_tokens_details': None, 'prompt_time': 0.000445782, 'prompt_tokens_details': None, 'qu

### 2. Simple RAG function: retrieve context + generate response

In [27]:
def rag_simple(query, retriever, llm, top_k=3):
    ## retrive the context
    results = retriever.retriever(query, top_k=top_k)

    context = "\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    ## generate the answer using GROQ LLM
    prompt=f"""Use the following comtext to answer the question concisely.
    Context:
    {context}

    Question: {query}
    
    Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])

    return response.content

In [28]:
answer = rag_simple("What is mlops", rag_retriever, llm)

Retrieving documents for query: 'What is mlops'
Top k: 3, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 10.28it/s]

Generated embeddings with shape: (1, 384)
Retrived 3 documents (after filtering)


In [32]:
print(answer)


<think>
Here's a thinking process:

1.  **Analyze User Input:**
   - **Context:** Repeated paragraphs explaining the relationship between MLOps and ML systems design. Key points:
     - "Ops in MLOps comes from DevOps, short for Developments and Operations."
     - "To operationalize something means to bring it into production, which includes deploying, monitoring, and maintaining it."
     - "MLOps is a set of tools and best practices for bringing ML into production."
     - ML systems design takes a system approach to MLOps, considering the ML system holistically.
   - **Question:** What is mlops
   - **Constraint:** Answer concisely based on the context.

2.  **Extract Key Information from Context:**
   - MLOps stands for Machine Learning Operations (implied by "Ops in MLOps comes from DevOps").
   - It is defined as: "a set of tools and best practices for bringing ML into production."
   - Operationalizing means deploying, monitoring, and maintaining it.

3.  **Formulate Concise A

# Enhanced RAG Pipeline Features

In [ ]:
def rag_advance(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optinally full context.
    """
    results = retriever.retriever(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}

    # prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext: \n{context}\n\nQuestion: {query}\n\nAnswer:"""